In [2]:
import torch
import numpy as np
import os
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.amp import autocast
from transformers import VideoMAEModel
from utils.dataset import WLASLDataset

In [ ]:
DEVICE     = torch.device("cuda")
NUM_FRAMES = 16
JSON_FILE  = ""
VIDEO_ROOT = ""
SAVE_ROOT  = ""
os.makedirs(SAVE_ROOT, exist_ok=True)


In [ ]:
backbone = VideoMAEModel.from_pretrained(
    "CHANGE TO YOUR OWN PATH",
    local_files_only=True
).to(DEVICE).eval()

for split in ['train', 'val', 'test']:
    dataset = WLASLDataset(JSON_FILE, VIDEO_ROOT, split=split,
                           num_frames=NUM_FRAMES)
    loader  = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)

    for i, (video, labels) in enumerate(tqdm(loader, desc=f"Extracting {split}")):
        video = video.to(DEVICE)
        video = video.permute(0, 2, 1, 3, 4)  # [B,C,T,H,W] → [B,T,C,H,W]

        with torch.no_grad():
            with autocast('cuda'):
                out = backbone(pixel_values=video)

        # last_hidden_state: [B, 1568, 768]
        feat = out.last_hidden_state  # [B, 1568, 768]
        B    = feat.shape[0]

        # reshape → [B, 8, 196, 768] → mean over spatial → [B, 8, 768]
        feat = feat.view(B, 8, 196, 768).mean(dim=2)  # [B, 8, 768]
        feat = feat.cpu().numpy()

        for j in range(B):
            idx    = i * loader.batch_size + j
            if idx >= len(dataset):
                break
            vid_id = dataset.video_ids[idx]
            label  = labels[j].item()
            np.save(
                os.path.join(SAVE_ROOT, f"{vid_id}.npy"),
                {'feature': feat[j], 'label': label}  # [8, 768]
            )

print("Done!")

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: /home/haod6/assignment3/model/vit/ssv2
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 
fc_norm.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TRAIN] 1897 videos | 300 classes


Extracting train:   0%|          | 0/119 [00:00<?, ?it/s]

[h264 @ 0x623a1e3e5740] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x623a1e3e5740] missing picture in access unit with size 10780
[h264 @ 0x623a1efb9c40] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x623a1efb9c40] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x623a1edca240] stream 1, offset 0x2a27a7: partial file
[h264 @ 0x623a1e4d7240] Invalid NAL unit size (745 > 472).
[h264 @ 0x623a1e4d7240] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x623a1ede3b80] stream 1, offset 0x3b468: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x623a1ede3b80] stream 1, offset 0x3b7d3: partial file


[VAL] 446 videos | 300 classes


Extracting val:   0%|          | 0/28 [00:00<?, ?it/s]

[TEST] 317 videos | 300 classes


Extracting test:   0%|          | 0/20 [00:00<?, ?it/s]

Done!
